In [1]:
import anthropic
import json
import os
import asyncio
from typing import Callable

In [2]:
client = anthropic.Anthropic()

In [3]:
def chat(messages):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=messages
    )
    return response.content[0].text

def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

In [4]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=5):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(self, task_description, prompt_inputs_spec, output_file, num_cases=5):
        spec_str = json.dumps(prompt_inputs_spec, indent=2)
        messages = []
        add_user_message(messages, f"""Generate {num_cases} diverse test cases for this task:
{task_description}

Each test case should have these fields:
{spec_str}

Return ONLY a raw JSON array with no markdown, no explanation, no code fences.""")
        response = chat(messages)
        clean = response.strip()
        if clean.startswith("```"):
            clean = clean.split("\n", 1)[-1]
            clean = clean.rsplit("```", 1)[0]
        dataset = json.loads(clean.strip())
        with open(output_file, 'w') as f:
            json.dump(dataset, f, indent=2)
        return dataset

    def grade_output(self, prompt_inputs, output, extra_criteria=""):
        messages = []
        add_user_message(messages, f"""Grade this AI output on a scale of 1-10.

Input given to the AI:
{json.dumps(prompt_inputs)}

AI Output:
{output}

Criteria:
{extra_criteria}

Respond with:
SCORE: <1-10>
REASONING: <why>""")
        response = chat(messages)
        lines = response.strip().split('\n')
        raw_score = lines[0].replace('SCORE:', '').strip().split('/')[0].strip()
        score = float(raw_score)
        reasoning = '\n'.join(lines[1:]).replace('REASONING:', '').strip()
        return {'score': score, 'reasoning': reasoning}

    async def _evaluate_single(self, semaphore, run_prompt_function, case, extra_criteria):
        async with semaphore:
            loop = asyncio.get_event_loop()
            output = await loop.run_in_executor(None, run_prompt_function, case)
            result = await loop.run_in_executor(None, self.grade_output, case, output, extra_criteria)
            result['output'] = output
            result['inputs'] = case
            return result

    async def run_evaluation(self, run_prompt_function, dataset_file, extra_criteria=""):
        with open(dataset_file) as f:
            dataset = json.load(f)

        semaphore = asyncio.Semaphore(self.max_concurrent_tasks)
        tasks = [
            self._evaluate_single(semaphore, run_prompt_function, case, extra_criteria)
            for case in dataset
        ]
        results = await asyncio.gather(*tasks)
        scores = [r['score'] for r in results]
        avg_score = sum(scores) / len(scores)
        print(f"Average Score: {avg_score:.2f}/10")
        print(f"Individual scores: {scores}")
        return results

In [5]:
evaluator = PromptEvaluator(max_concurrent_tasks=5)

dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3
)

print(json.dumps(dataset, indent=2))

[
  {
    "height": "185",
    "weight": "90",
    "goal": "Build muscle mass and increase strength",
    "restrictions": "None"
  },
  {
    "height": "165",
    "weight": "58",
    "goal": "Lose body fat while maintaining endurance for marathon training",
    "restrictions": "Vegetarian, lactose intolerant"
  },
  {
    "height": "175",
    "weight": "72",
    "goal": "Improve recovery and maintain current physique during competition season",
    "restrictions": "Gluten-free, no shellfish"
  }
]


In [6]:
def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [7]:
results = await evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown  
- Meals with exact foods, portions, and timing
"""
)

Average Score: 5.67/10
Individual scores: [5.0, 6.0, 6.0]


In [8]:
for i, r in enumerate(results):
    print(f"Case {i+1} | Score: {r['score']}/10")
    print(r['reasoning'])
    print("---")

Case 1 | Score: 5.0/10
The output partially meets the criteria but falls short in key areas:

**What it does well:**
- Provides a clear daily caloric total (3,300-3,500 calories) ✅
- Includes a solid macronutrient breakdown with grams and calories in a clean table format ✅
- Formatting is clean, readable, and professional
- Includes useful contextual information like BMI and TDEE estimates

**Where it fails:**
- **No exact portions** — Food choices are listed generically (e.g., "chicken breast, eggs, beef") with zero gram/ounce measurements, which makes the plan difficult to actually follow ❌
- **No specific meals** — There is no structured meal plan (e.g., Meal 1, Meal 2, etc.) with defined foods per meal ❌
- **No timing structure** — While it mentions eating every 3-4 hours and pre/post workout nutrition, it never maps this to an actual daily schedule or assigns foods to specific time slots ❌

**Summary:** The output reads more like a general reference guide than a personalized, acti